# Tam Eğitim Notebook'u (A100) — Türkçe OCR İnce Ayarı

Bu notebook, **A100 GPU** ile tam veri ve tam epoch sayısıyla asıl ince ayarı çalıştırır.
Buraya geçmeden ÖNCE `00_pilot_t4_setup_eda.ipynb`'nin T4'te HATASIZ tamamlandığından
emin olun — pilot, bu notebook'un güvenle çalışacağının kanıtıdır.

Çalıştırmadan önce Colab menüsünden: **Çalışma zamanı > Çalışma zamanı türünü değiştir >
A100 GPU** seçili olmalıdır (Colab Pro/Pro+ gerektirir).

Hücreleri SIRAYLA çalıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/nidazeren/qwen2.5-vl.git"
REPO_DIR = "/content/qwen2.5-vl"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## (İsteğe bağlı) flash-attention-2 kurulumu
A100'de flash-attention-2 kuruluysa `training/lora_setup.py` otomatik olarak onu
seçer (bkz. `select_dtype_and_attn_impl`); kurulu değilse otomatik olarak `sdpa`'ya
(daha yavaş ama tamamen doğru/güvenli) düşer. Derlemesi birkaç dakika sürebilir; hız
kazancı istemiyorsanız bu hücreyi atlayabilirsiniz.

In [ ]:
!pip install -q flash-attn --no-build-isolation

## Ortam değişkenleri: TAM MOD (PILOT_MODE=0)
Bu, `configs/config.py` içindeki tüm boyutları (veri, epoch, batch) tam-ölçekli
değerlere geçirir VE çıktı klasörlerini (`processed_data/full`, `checkpoints/full`,
`eval_outputs/full`) pilot çıktılarından AYRI tutar (bkz. config.py: MODE_TAG).

In [ ]:
import os, sys

os.environ["QWEN_OCR_PILOT_MODE"] = "0"
os.environ["QWEN_OCR_DRIVE_ROOT"] = "/content/drive/MyDrive/qwen25vl_turkish_ocr"
sys.path.insert(0, REPO_DIR)

from configs import config
config.ensure_directories()
print("PILOT_MODE:", config.PILOT_MODE)
print("MODE_TAG:", config.MODE_TAG)
print("Hedef kova boyutlari (tam):", config.compute_bucket_target_sizes())

## Kaggle kimlik bilgisi
Pilot notebook'ta bir kez Drive'a kaydettiyseniz burada otomatik bulunur.

In [ ]:
import os, shutil, stat

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
target = os.path.join(kaggle_dir, "kaggle.json")
drive_copy = str(config.DRIVE_ROOT / "kaggle.json")

if os.path.exists(drive_copy):
    shutil.copy(drive_copy, target)
elif not os.path.exists(target):
    from google.colab import files
    print("Lutfen kaggle.json dosyanizi secin:")
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, target)
    shutil.copy(target, drive_copy)

os.chmod(target, stat.S_IRUSR | stat.S_IWUSR)
print("Kaggle kimlik bilgisi hazir:", target)

## Tam veri hazırlama
`PILOT_MODE=False` olduğundan `data/prepare_datasets.py`, `RAW_DATA_DIR` altındaki
(pilotta küçük boyutlu kaydedilmiş) kaynakları TAM boyutlarıyla YENİDEN indirir/kaydeder.

In [ ]:
!python data/prepare_datasets.py

## (İsteğe bağlı) EDA'yı tam veriyle tekrar gözden geçirin

In [ ]:
!python analysis/tokenizer_analysis.py
!python analysis/vision_token_eda.py

## Self-distillation replay verisi (tam boyut)
`data/replay_generation.py`, mevcut replay verisinin YENİ (tam) hedefleri karşılayıp
karşılamadığını kontrol eder; karşılamıyorsa (pilotun küçük verisi yetersiz kalacağından)
otomatik olarak yeniden üretir. Bu adım A100'de bile en uzun süren adımlardan biri olabilir.

In [ ]:
!python data/replay_generation.py

In [ ]:
!python data/build_chat_dataset.py

## Baseline değerlendirme (tam Test A/B setleri üzerinde)
MODE_TAG=full olduğundan bu, pilotun baseline.json'ından TAMAMEN AYRI, tam Test A/B
setlerine göre yeni bir baseline üretir/kullanır.

In [ ]:
!python evaluation/evaluate.py --tag baseline

## Tam LoRA eğitimi
`config.NUM_TRAIN_EPOCHS` (varsayılan 3), her epoch sonunda Test A/B otomatik
değerlendirilir (training/callbacks.py). Test A'da belirgin bir kötüleşme tespit
edilirse eğitim OTOMATİK DURUR ve `eval_outputs/full/regression_report.json` içine
önerilen sonraki adımlar yazılır (LR düşürme / karışım oranı değiştirme).

In [ ]:
!python training/train_sft.py

## Sonuçları incele

In [ ]:
import json

for path in sorted(config.EVAL_OUTPUT_DIR.glob("*.json")):
    print(f"--- {path.name} ---")
    print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    print()

## Eğer regresyon nedeniyle eğitim durduysa: yeniden başlatma
1) `eval_outputs/full/regression_report.json` içindeki önerileri okuyun.
2) `configs/config.py` içinde `LEARNING_RATE` ve/veya `MIXTURE_REPLAY`/`MIXTURE_HANDWRITING`
   değerlerini güncelleyin (repoyu düzenleyip GitHub'a push edip tekrar `git pull` ile
   Colab'a çekebilir, ya da doğrudan Colab'da dosyayı düzenleyip devam edebilirsiniz).
3) `training/train_sft.py`'yi tekrar çalıştırın; `SFTConfig(save_strategy="epoch")`
   sayesinde `config.CHECKPOINT_DIR / "trainer_output"` altında checkpoint'ler mevcuttur,
   isterseniz `trainer.train(resume_from_checkpoint=True)` ile devam edecek şekilde
   `training/train_sft.py` içindeki `trainer.train()` çağrısını uyarlayabilirsiniz.